## Instrucciones y Ejemplos

### Base

In [7]:
instr_base = """
### OBJETIVO PRINCIPAL
Eres un asistente experto en derecho registral chileno, especializado en la extracción y estructuración de datos de 
inscripciones de derechos de aprovechamiento de aguas. Tu objetivo es leer el texto de un documento emitido por un 
Conservador de Bienes Raíces (CBR) que ya ha sido pre-procesado con etiquetas (ej. [Cuerpo central:]). Debes identificar 
con precisión los campos clave y estructurarla en el formato JSON de salida especificado.

### INSTRUCCIONES GENERALES
1.  **Análisis del Documento:**
        - Lee detenidamente todo el texto proporcionado, prestando especial atención a las etiquetas que permitirán precisar 
        dónde se encuentra los campos que se te solicitan.

2.  **Regla Global:** 
        - Si cualquier dato numérico (años, fojas, números de inscripción, fechas, valores, caudales o coordenadas UTM/latitud/longitud) 
          está escrito en palabras, debes extraerlo convertido a número.
        
3.  **Extracción Precisa:** 
        - Identifica cada pieza de información correspondiente a los campos del modelo JSON de salida.
        - La información SIEMPRE debe provenir del documento, debes procurar la mayor fidelidad posible.
        - Algunos de los campos solicitados pueden no ser mencionados de forma explícita, pero debes ser 
          capaz de actuar como un experto e inferir dichos campos usando la información implícita de de manera inteligente 

4.  **Formato de Salida Obligatorio:** 
        - Tu respuesta final SIEMPRE debe seguir estrictamente la estructura solicitada.
"""


### Modelo 1: Datos Basales

In [17]:
examples_1 = [
    {
        "input": """
            [inscripción / Carátula:]
            CBRS - Conservador de Bienes Raíces de Santiago
            Registro de Propiedad de Aguas
            Carátula Nº 23814355
            Copia de Inscripción
            
            El Conservador de Bienes Raíces de Santiago certifica que la inscripción de fojas
            208 número 231 correspondiente al Registro de Propiedad de Aguas del año 2025,
            adjunta al presente documento, está conforme con su original.
            
            Los derechos registrales correspondientes a esta actuación ascienden a la suma de $ 2.600.
            Santiago, 3 de junio de 2025. (Fecha de emisión de copia)
            Firmado digitalmente por: cvasquez
            Código de verificación: asv16b60d3-27-769-0
            
            --- Nueva Página ---
            
            [Nota al margen izquierdo:]
            Nº 231
            HERENCIA
            SERGIO IDELFONSO ALBA CORRAL
            A
            ALEJANDRA ANDREA ALBA FERNANDEZ Y OTROS
            REP: 39733
            
            [Cuerpo central:]
            Santiago, dos de mayo del año dos mil veinticinco.
            Doña ALEJANDRA ANDREA ALBA FERNANDEZ, C.I. 13.055.xxx-x, doña JAVIERA OLGA ALBA FERNANDEZ... 
            son dueños de derechos de aprovechamiento de aguas, que consisten en cero coma cero
            veintisiete mil seiscientas veinticinco acciones de aguas del canal Las Mercedes...
            que adquirieron por herencia de don SERGIO IDELFONSO ALBA CORRAL, según resolución
            que consta en la inscripción de fojas 206 número 227 del año 2025.
            
            El título anterior está a fojas 19 número 26 de 2011.
            Requirente: Iván González López.
            """,
        "output": {
            "tipo_transaccion": "HERENCIA",
            "numero_cbr_actual": 231,
            "fojas_cbr_actual": "208",
            "dia_cbr_actual": 2,    
            "mes_cbr_actual": 5,
            "ano_cbr_actual": 2025
        }
    },
    {
        "input": """
            [Resumen inscripción:]
            Registro de Propiedad de Aguas Fs 150 vta Nro 205-2023
            Conservador de Bienes Raíces de LINARES
            ...
            [Nota al margen izquierdo:]
            Nº 205
            COMPRAVENTA
            AGRICOLA LOS PIMPPOLLOS LTDA
            ...
            [Cuerpo central:]
            Linares, quince de marzo del año dos mil veintitrés.
            Comparece don...
            """,
        "output": {
            "tipo_transaccion": "COMPRAVENTA",
            "numero_cbr_actual": 205,
            "fojas_cbr_actual": "150v",
            "dia_cbr_actual": 15,
            "mes_cbr_actual": 3,
            "ano_cbr_actual": 2023
        }
    },
    {
        "input": """
            [Resumen inscripción:]
            Registro Propiedad Aguas
            Fojas 500 Número 300 Año 2010
            ...
            [Nota al margen izquierdo:]
            Nº 300
            PROPIEDAD
            ...
            [Cuerpo central:]
            En Talca, a veinte de diciembre de dos mil diez.
            Ante mí comparecen don Juan Pérez y doña María Soto,
            quienes exponen: Que por este acto venden, ceden y transfieren la 
            totalidad de los derechos de aprovechamiento de aguas a 
            Inversiones El Roble SpA, quien compra y adquiere para sí...
            """,
        "output": {
            "tipo_transaccion": "COMPRAVENTA",
            "numero_cbr_actual": 300,
            "fojas_cbr_actual": "500",
            "dia_cbr_actual": 20,
            "mes_cbr_actual": 12,
            "ano_cbr_actual": 2010
        }
    }
]

instr_fase1 = """
### OBJETIVO ESPECÍFICO: Extracción de Datos de Inscripción Actual e Inferencia de Transacción

Debes extraer los metadatos del registro y descomponer la fecha en día, mes y año numéricos. Además, debes determinar el `tipo_transaccion` normalizado.

### 1. DATOS DE REGISTRO (Fojas y Número)
   - **Fuente:** Etiqueta **[Resumen inscripción:]** o encabezado superior.
   - **Patrón:** "Registro de Propiedad de Aguas Fs [FOJAS] Nro [NUMERO]-[AÑO]".
   - **Salida:** - `fojas_cbr_actual` es un string que usa "bis" o "v" (en lugar de "vta" si existe).
     - `numero_cbr_actual` (int).

### 2. FECHA DE INSCRIPCIÓN (Día, Mes, Año)
   - **Fuente:** Primera etiqueta **[Cuerpo central:]** (usualmente tras "--- Nueva Página ---").
   - **Patrón:** Busca la fecha protocolar al inicio del texto: "[Ciudad], a [Día] de [Mes] del Año [Año].-".
   - **Acción:** Convierte las palabras a números enteros (ej: "quince" -> 15, "marzo" -> 3).
   - **Salida:**
     - `dia_cbr_actual` (int): El día del mes (1-31).
     - `mes_cbr_actual` (int): El mes del año (1-12).
     - `ano_cbr_actual` (int): El año (ej: 2023). 
   ⚠️ **ADVERTENCIA CRÍTICA:** Los documentos pueden empezar con una "Carátula" o "Certificado de Vigencia" que tiene una fecha reciente (ej: fecha de impresión). 
       **IGNORA** cualquier fecha que esté cerca de frases como:
         * "El Conservador certifica..."
         * "Copia conforme con su original..."
         * "Los derechos registrales ascienden a..."
     
### 3. TIPO DE TRANSACCIÓN

**PASO A: Búsqueda Explícita (Margen)**
   - Busca en **[Nota al margen izquierdo:]** bajo el "N°".
   - Si encuentras un término claro que coincida con la lista permitida (ej: "COMPRAVENTA", "HERENCIA", "REGULARIZACIÓN"), úsalo.
   - **Ignorar:** Si dice solo "PROPIEDAD", "AGUAS", "CO", o "INSCRIPCIÓN", ignóralo y pasa al PASO B.

**PASO B: Inferencia por Contexto (Cuerpo Central)**
   - Si el margen no es claro, lee el acto jurídico en el **[Cuerpo central:]** e infiere el tipo exacto de la lista `TipoTransaccion`.
   - **Guía de Inferencia:**
     * **COMPRAVENTA:** "vende, cede y transfiere", "compra y adquiere".
     * **POSESIÓN EFECTIVA:** "auto de posesión efectiva", "se concede la posesión efectiva".
     * **HERENCIA:** "adjudicación por herencia", "fallecimiento de", "herederos de".
     * **REGULARIZACIÓN:** "Artículo 2° Transitorio", "Regularización de derechos".
     * **PERFECCIONAMIENTO:** "Perfeccionamiento", "completa características (caudal/tiempo)".
     * **DONACIÓN:** "donación gratuita", "cede a título gratuito".
     * **APORTE:** "aporta a la sociedad", "constitución de sociedad".
     * **REMATE:** "acta de remate", "adjudicación en remate".
     * **SERVIDUMBRE:** "constitución de servidumbre", "gravamen".
     * **HIPOTECA:** "constituye hipoteca".
     * **USO DE SUELOS / CAMBIOS:** Si implica cambio de punto de captación, usa "CAMBIO DE PUNTO DE CAPTACIÓN".
     * **RESOLUCIONES DGA/SAG:** Si el origen es administrativo, usa "RESOLUCIÓN" o "RESOLUCIÓN SAG".

"""

### Modelo 2: Caracterización Derecho

In [9]:
examples_2 = [
    {
        "input": """
            -....
            Santiago, veintiuno de febrero de dos mil doce... 
            Doña NANCY SOLEDAD RODRIGUEZ GALVEZ... es dueña de derechos de aprovechamiento de aguas subterráneas de ejercicio permanente y continuo 
            de cero coma seiscientos dos litros por segundo del pozo ubicado en la Parcela cuarenta y dos A del Proyecto de Parcelación Chacabuco-Nuevo Colorado...
            Los adquirió por compra a la sociedad AGROPARCELAS CHILE SpA...
            [Nota al margen izquierdo:]
            N°71... COMPRAVENTA...
            """,
        "output": {
            "tipo_derecho": ["CONSUNTIVO"],
            "naturaleza_agua": ["SUBTERRÁNEA"],
            "ejercicio_derecho": ["PERMANENTE Y CONTINUO"],
            "uso_agua": ["AGRICULTURA"]
        }
    },
    {
        "input": """
            [Cuerpo central:]
            Rancagua, quince de marzo del año dos mil veinticuatro.- La sociedad "GENERADORA HIDROELÉCTRICA TAPIHUE SpA", RUT 99.551.672-K, es dueña de dos derechos de aprovechamiento de aguas superficiales y corrientes, ambos de carácter no consuntivo, captadas gravitacionalmente desde la ribera sur del Río Claro, comuna de Rengo.
            El primero de éstos es un derecho de ejercicio permanente y continuo, por un caudal máximo de quince metros cúbicos por segundo.
            El segundo de éstos es un derecho de ejercicio eventual y continuo, que se ejercerá únicamente cuando el caudal matriz supere los niveles medios, por un caudal de diez metros cúbicos por segundo.
            Las aguas de ambos derechos serán captadas en la bocatoma "El Sauce" definida por las coordenadas UTM Norte seis millones... y serán restituidas íntegramente al cauce del mismo Río Claro, aguas abajo, en el punto de descarga de la Sala de Máquinas de la Central Hidroeléctrica "Los Cipreses", sin alterar la calidad del recurso.- Título: Resolución DGA número...
            """,
        "output": {
            "tipo_derecho": ["NO CONSUNTIVO", "NO CONSUNTIVO"],
            "naturaleza_agua": ["SUPERFICIAL Y CORRIENTE", "SUPERFICIAL Y CORRIENTE"],
            "ejercicio_derecho": ["PERMANENTE Y CONTINUO", "EVENTUAL Y CONTINUO"],
            "uso_agua": ["GENERACIÓN ELÉCTRICA", "GENERACIÓN ELÉCTRICA"]
        }
    }
]

instr_fase2 = """
### OBJETIVO ESPECÍFICO: Extracción de Atributos del Derecho

Debes analizar el texto contenido principalmente bajo la etiqueta **[Cuerpo central:]** (y sus continuaciones en páginas siguientes) para extraer las características técnicas del derecho de agua.

### REGLAS DE EXTRACCIÓN Y MAPEO:
"...es dueño de derechos de agua..." o "...son dueños de derechos de agua..." o "...es dueña de derechos de agua...".

**1. DETECCIÓN MULTIPLE (Sincronización):**
   - Si el texto enumera múltiples propiedades que incluyen aguas (ej. "Parcela 1 con sus aguas... y Parcela 2 con derechos en el canal..."), debes crear una entrada por cada uno de los derechos distintos identificados.
   - **Regla de Oro:** Todas las listas de salida (`tipo_derecho`, `naturaleza_agua`, `ejercicio_derecho`, `uso_agua`) siempre deben tener EXACTAMENTE el mismo largo. Si un derecho tiene un dato desconocido, usa "NO INDICA/OTRO" en esa posición.

**2. 'tipo_derecho'**
   - Busca: "Consuntivo" o "No Consuntivo".
   - **Inferencia:** - Si se trata de "Acciones de Canal", asume "CONSUNTIVO".
     - Si es un "Pozo", "regadores" o agua subterránea destinada a riego/consumo y no se especifica, asume "CONSUNTIVO".
     - Cuando se trate de hidroeléctricas es probable que sea "NO CONSUNTIVO".
     - Si solo dice "Derechos de agua" sin más detalles que permitan inferir su tipo de manera clara, usa "NO INDICA/OTRO".

**3. 'naturaleza_agua'**
   - Clasifica según la fuente:
     - **SUPERFICIAL Y CORRIENTE:** Ríos, Esteros, Canales, Vertientes, Quebradas, "Aguas corrientes".
     - **SUPERFICIAL Y DETENIDA:** Tranque, Embalse, Laguna, Lago.
     - **SUPERFICIAL CORRIENTES/DETENIDAS:** Se explicita que es Superficial, pero no queda claro si es corriente o detenida "Aguas del predio"
     - **SUBTERRÁNEA:** Pozo, Noria, Puntera, Sondaje, "Aguas subterráneas".

**4. 'ejercicio_derecho'**
   - Busca combinaciones: "Permanente y Continuo", "Eventual", "Alternado".
   - **Inferencia:** Si se mencionan "Acciones" (de un canal), el estándar suele ser "PERMANENTE Y CONTINUO". 
   - Si solo dice "Permanente" y no aclara si es continuo/alternado $\rightarrow$ PERMANENTE Y NO ESPECIFICADO
   - Si solo dice "Eventual" y no aclara continuo/alternado $\rightarrow$ EVENTUAL Y NO ESPECIFICADO. 
   - Si no hay pistas, usa "NO INDICA/OTRO".

**5. CAMPO: 'uso_agua'**
   - **Inferencia por Contexto:** - Si se mencionan palabras cómo: "Parcela", "Fundo", "Viña", "Agrícola", "Riego", "Cultivo", "Huerto", "Canal de regadío" -> Asume **"AGRICULTURA"**, que es el campo más común
     - Si menciona "Bebida", "Doméstico", "Casa" -> Asume **"USO DOMÉSTICO"** o "AGUA POTABLE".
     - Si menciona "Industrial", "Fábrica" -> Asume **"INDUSTRIA"**.
   - Si no hay contexto del destino del agua, usa "NO INDICA/OTRO".
"""

### Modelo 3: Personas

In [2]:
naturaleza = ''
examples_3 = [
    {
        "input": """
            [Cuerpo central:]
            [Manuscrita: De conformidad a subinscripción... 
            Doña María Magdalena Marambio Ureta; 
            Genny Jacqueline Marambio Marambio 
            y doña Laura del Tránsito Marambio Marambio; son dueños de los derechos... 
            Adquirieron estos derechos por herencia intestada quedada al fallecimiento de don JOSE OCTAVIO MARAMBIO SORIA.
            Y venden, ceden y transfieren a Agrícola El Cantal Limitada RUT 76.481.932-K para y por quien acepta su representante 
            don Fernando Ernesto Prado Echeverría, C.I. 12.542.944-2, la totalidad de los derechos... Melipilla 20 de diciembre de 2013.-]
            """,
        "output": {
            "tipo_persona": ["EMISOR", "EMISOR", "EMISOR", "RECEPTOR"],
            "rut": ["NO INDICA/OTRO", "NO INDICA/OTRO", "NO INDICA/OTRO", "76.481.932-K"],
            "nombre": ["MARÍA MAGDALENA MARAMBIO URETA", "GENNY JACQUELINE MARAMBIO MARAMBIO", "LAURA DEL TRÁNSITO MARAMBIO MARAMBIO", "AGRÍCOLA EL CANTAL LIMITADA"],
            "nombre_representante_legal": ["NO INDICA/OTRO", "NO INDICA/OTRO", "NO INDICA/OTRO", "FERNANDO ERNESTO PRADO ECHEVERRÍA"],
            "rut_representante_legal": ["NO INDICA/OTRO", "NO INDICA/OTRO", "NO INDICA/OTRO", "12.542.944-2"]
        }
    },
    {
        "input": """
            [Cuerpo central:]
            Valparaíso, 15 de enero de 2024.
            Se ha procedido a la regularización del derecho de aprovechamiento de aguas, conforme al artículo 2° transitorio del Código de Aguas.
            Se inscribe a nombre de SOCIEDAD DE INVERSIONES LOS ALERCES SPA, R.U.T. 76.555.434-K, representada por don JUAN PÉREZ COTAPOS, C.I. 12.345.678-9, ambos con domicilio en Calle Prat 456, Valparaíso.
            El derecho consiste en un caudal de dos litros por segundo consuntivos ....
            """,
        "output": {
            "tipo_persona": ["RECEPTOR"],
            "rut": ["76.555.434-K"],
            "nombre": ["SOCIEDAD DE INVERSIONES LOS ALERCES SPA"],
            "nombre_representante_legal": ["JUAN PÉREZ COTAPOS"],
            "rut_representante_legal": ["12.345.678-9"],
            "direccion": ["CALLE PRAT 456, VALPARAÍSO"]
        }
    }
]

instr_fase3 = f"""
### OBJETIVO ESPECÍFICO: Identificación de Personas y Partes Involucradas en el Acto Jurídico. Es posible que se mencionen OTRAS personas en el documento,
por ejemplo, de compraventas pasadas del derecho. Lo relevante son las partes escenciales del documento. 

### REGLA DE ORO: LISTAS PARALELAS
El formato de salida requiere listas (`[]`). **Cada entrada en las listas debe corresponder a la misma persona en la misma posición (índice).**
* Si encuentras 3 personas, todas las listas (`tipo_persona`, `rut`, `nombre`, `representante`, etc.) deben tener largo 3.
* Si un dato no existe para una persona (ej. no tiene representante), usa "NO INDICA/OTRO" en esa posición para mantener la alineación.

### PASO 1: Determinar Personas y Roles según la Naturaleza del Acto
[[naturaleza]]

### PASO 2: Extracción de Campos

**A. 'nombre' y 'tipo_persona'**
   - Extrae el nombre completo o razón social en MAYÚSCULAS. 
   - Asigna "EMISOR" o "RECEPTOR" según el análisis del Paso 1.
   - Elimina títulos como "Don", "Doña", "Sociedad", "Señores".

**B. 'rut'**
   - Busca el RUT asociado a *esa* persona específica, que aparecerá por ejemplo como:
           - **"cédula nacional de identidad N°"**, **"C.I."**, **"C:"**, **"Rol Único Tributario Nº"**, **"Run N°"**, etc.
   - Formato: "X.XXX.XXX-X" o "XX.XXX.XXX-X" (sólo cuando antes del primer punto hay 2 dígitos, cómo en 22.840.174-2
   - El formato debe respetar estrictamente la posición de puntos y guión de separación del dígito verificador de un RUT chileno. 
   - Si el RUT está sin dígito verificador, marcar como NO INDICA/OTRO.
   - Si no aparece cerca del nombre, usa "NO INDICA/OTRO".

**C. 'nombre_representante_legal' y 'rut_representante_legal'**
   - Busca frases conectadas al nombre principal: "representada por", "en representación de", "a través de su apoderado", "por quien acepta don/doña", "en nombre de".
   - Extrae el nombre y RUT del representante en la posición correspondiente a la empresa/persona representada.
   - Asegúrate de no confundir el RUT del representante con el RUT de la empresa o del representado.
   - Si un sujeto tiene dos o más representantes, no crees nuevas posiciones en la lista. 
   - Agrupa todos los nombres en la misma posición de la lista, separados por el conector " Y " (ej. ['Juan Pérez Y María Jara', 'null']).
   - Si es una persona natural actuando por sí misma, Omite esta variable con 'nan', es Opcional. 
   - Para el rut_representante_legal sigue el mismo formato que 'rut'

**D. 'direccion'**
   - Busca "domiciliado en", "con domicilio en la comuna de"; para cada una de las partes o sus representantes. 
   - Entrega la dirección ordenada idealmente en formato 'Calle' 'Número', 'Comuna' (es posible que no todos estos elementos se encuentren).
   - NO escribas la Región, incluso si se identifica. 
   - Si no encuentras ninguna dirección, omite esta variable, es Opcional.
   - Asegura de no extraer la dirección del Conservador de bienes raíces. 

**Es fundamental mantener igual largo de todas las listas**
"""

diccionario_naturaleza = {
    f"NO INDICA/ERROR": """ """,
    f"RENUNCIA O LIMITACIÓN": """
    - **Lógica:** El o los titulares restringen su propio derecho.
    - **Textos Clave:** 'Renuncia a', 'Se reserva el derecho de', 'Alzó la prohibición/hipoteca', 'Pactan indivisión de', 'Prohíbe enajenar'.
    - **EMISOR:** El o los titulares (pueden ser varios) que realizan la renuncia o sobre quienes recae la limitación.""",

    f"MODIFICACIÓN TÉCNICA": """
    - **Lógica:** El o los mismos dueños cambian características técnicas.
    - **Textos Clave:** "Autorizase el cambio de punto de captación", "Traslada el ejercicio del derecho", "Rectifica la inscripción", "Perfecciona el derecho".
    - **EMISOR:** El o los titulares actuales del derecho (solicitantes o requirentes) que está modificando su forma de ejercicio. Si son varios dueños del derecho, lístalos a todos.""",

    f"ACTO DE AUTORIDAD O REGISTRO": """
    - **Lógica:** Reconocimiento de propiedad a una o varias personas.
    - **Textos Clave:** "Inscríbase a nombre de", "Se regulariza el derecho a favor de", "La sentencia declara", "La Resolución administrativa reconoce a", "Inscripción de dominio de".
    - **EMISOR:** Todas las personas (naturales o jurídicas) a cuyo nombre queda el derecho o que son reconocidas como dueños.""",

    f"TRASPASO": """
    - **Lógica:** El derecho pasa de un grupo de personas a otro.
    - **Verbos Clave:** "Venden", "Ceden", "Transfieren", "Permutan", "Dan en pago", "Expropian", "Rematan", "Rescilían".
    - **EMISOR:** El o los dueños anteriores (vendedores, cedentes, etc.). Si venden 3 personas, lístalas a las tres como EMISOR.
    - **RECEPTOR:** El o los dueños actuales (compradores, cesionarios, etc.). Si compran 2 personas, lístalas a ambas como RECEPTOR.""",

    f"REORGANIZACIÓN EMPRESARIAL": """
    - **Lógica:** Traspaso entre una o varias sociedades.
    - **Verbos Clave:** "Aportan a la sociedad", "Se fusiona con", "Absorbidas por", "Asignan por división", "Se liquida la sociedad".
    - **EMISOR:** La o las sociedades que aportan, desaparecen o se dividen.
    - **RECEPTOR:** La o las sociedades que reciben el aporte, la sociedad absorbente o la nueva sociedad creada.""",

    f"GRAVÁMENES Y USO": """
    - **Lógica:** Uno o varios dueños entregan un derecho de uso a uno o varios beneficiarios.
    - **Verbos Clave:** "Arriendan", "Constituyen usufructo/hipoteca", "da servidumbre de paso a", "Queda sujeto a fideicomiso".
    - **EMISOR:** El o los dueños que permiten el gravamen sobre su derecho de propiedad.
    - **RECEPTOR:** El o los beneficiarios (arrendatarios, acreedor hipotecario o fiduciario, usufructuarios).""",

    f"SUCESIÓN": """
    - **Lógica:** Traspaso de derecho de un fallecido a múltiples herederos.
    - **Verbos Clave:** "Fallecimiento de",  "Quedada al deceso de",  "Sucesión de", "Herederos de", "Se adjudica por partición", "Fue adqurido por herencia a favor de"
    - **EMISOR:** El o los causantes (la persona fallecida) de quien emana el derecho.
    - **RECEPTOR:** TODOS los herederos o adjudicatarios mencionados que reciben el derecho.""",

    f"ACTOS DE VOLUNTAD": """
    - **Lógica:** Entrega gratuita de una o varias personas a otras.
    - **Verbos Clave:** "Donan", "Legan a favor de", "En su testamento disponen".
    - **EMISOR:** El o los Donantes o Testadores. (quien dispone del bien)
    - **RECEPTOR:** El o los Donatarios o Legatarios. (quien recibe el beneficio)"""
}

### Modelo 4: Caudal

In [16]:
examples_4 = [
    {
        "input": """
            [Cuerpo central:]
            ...La sociedad "GENERADORA HIDROELÉCTRICA CORDILLERA SpA", es dueña de derechos de aprovechamiento en el Río Claro.
            En primer lugar, un derecho de ejercicio permanente, por un caudal máximo de quince metros cúbicos por segundo.
            Por otro lado, un derecho de ejercicio eventual, que se ejercerá únicamente sobre los excedentes, por un caudal de diez metros cúbicos por segundo.
            Ambos captados en la Bocatoma El Sauce...
            """,
        "output": {
            "tipo_caudal": ["PERMANENTE", "EVENTUAL"],
            "unidad_caudal": "M3/S",
            "caudal_promedio": [15.0, 10.0],
            "tipo_fuente": ["RÍO / ESTERO", "RÍO / ESTERO"],
            "fuente": ["Bocatoma El Sauce", "Bocatoma el Sauce"],
            "cuenca": ["Río Claro", "Río Claro"], 
            "caudal_ene": None, "caudal_feb": None, "caudal_mar": None, 
            "caudal_abr": None, "caudal_may": None, "caudal_jun": None,
            "caudal_jul": None, "caudal_ago": None, "caudal_sep": None, 
            "caudal_oct": None, "caudal_nov": None, "caudal_dic": None
        }
    },
    {
        "input": """
            [Cuerpo central:]
            San Felipe, quince de marzo del año dos mil veintitrés.
            La sociedad "AGRÍCOLA E INVERSIONES LAS LILAS LIMITADA", RUT noventa y nueve millones quinientos ochenta y tres mil seiscientos cincuentaguión cinco, 
            es dueña de dos derechos de aprovechamiento de aguas superficiales y corrientes del Río Aconcagua, 
            captadas ambas en la bocatoma del Canal Comunero Nuevo Colorado, ubicada en la ribera sur, parcela cuarenta y dos A del Proyecto de Parcelación San Felipe-Nuevo Colorado.

            PRIMERO: Un derecho de aprovechamiento consuntivo, de ejercicio permanente y continuo, con una distribución estacional variable
            ("Turno A"), consistente en un caudal promedio anual referencial de setenta y seis coma seis litros por segundo, 
            distribuido bajo el siguiente régimen mensual:
            - Enero: ciento veinte litros por segundo;
            - Febrero: cien litros por segundo;
            - Marzo: ochenta litros por segundo;
            - Abril a Agosto: cincuenta litros por segundo en cada mes;
            - Septiembre: sesenta litros por segundo;
            - Octubre: ochenta litros por segundo;
            - Noviembre y Diciembre: cien litros por segundo.
            
            SEGUNDO: Un derecho de aprovechamiento consuntivo de ejercicio eventual y continuo ("Turno B"), 
            consistente en un caudal de diez litros por segundo, el cual solo podrá captarse cuando el caudal matriz del río asegure la satisfacción
            de los derechos permanentes de la Primera Sección del Río Aconcagua.
            
            Título: Adquiridos por compra a AGROPARCELAS CHILE SpA, según escritura pública de fecha dieciséis de diciembre de dos mil veintidós, 
            Notaría de San Felipe de doña Valeria Ronchera Flores.
            """,
        
        "output": {
            "tipo_caudal": ["PERMANENTE", "EVENTUAL"],
            "unidad_caudal": "L/S",
            "caudal_promedio": [76.6, 10.0],
            "tipo_fuente": ["CANAL", "CANAL"],
            "fuente": ["Canal Comunero Nuevo Colorado", "Canal Comunero Nuevo Colorado"],
            "cuenca": ["Aconcagua", "Aconcagua"], 
            "caudal_ene": [120.0, None],
            "caudal_feb": [100.0, None],
            "caudal_mar": [80.0, None],
            "caudal_abr": [50.0, None],
            "caudal_may": [50.0, None],
            "caudal_jun": [50.0, None],
            "caudal_jul": [50.0, None],
            "caudal_ago": [50.0, None],
            "caudal_sep": [60.0, None],
            "caudal_oct": [80.0, None],
            "caudal_nov": [100.0, None],
            "caudal_dic": [100.0, None]
        }
    },
    {
        "input": """
        [Cuerpo central:]
        ...es dueño de dos coma tres regadores del Canal Bellavista, provenientes del Río Teno, equivalentes a una parte alícuota en el caudal matriz...
        """,
        "output": {
            "tipo_caudal": ["PERMANENTE"],
            "unidad_caudal": "REGADORES",
            "caudal_promedio": [2.3],
            "tipo_fuente": ["CANAL"],
            "fuente": ["Canal Bellavista"],
            "cuenca": ["Río Teno"],
            "caudal_ene": None, "caudal_feb": None, "caudal_mar": None, 
            "caudal_abr": None, "caudal_may": None, "caudal_jun": None,
            "caudal_jul": None, "caudal_ago": None, "caudal_sep": None, 
            "caudal_oct": None, "caudal_nov": None, "caudal_dic": None
        } 
    }
]




instr_fase4= """
### OBJETIVO ESPECÍFICO: Extracción de Caudales, Unidades y Estacionalidad

Analiza el texto para extraer cifras de caudal, identificar la unidad de medida y capturar tablas de distribución mensual si existen.

### REGLAS DE EXTRACCIÓN Y MAPEO:

**REGLA DE ORO: LISTAS PARALELAS**
   - Debes generar una entrada en las listas `tipo_caudal`, `caudal_promedio`, `fuente` y `cuenca` por CADA UNO DE LOS derechos identificados.
   - Las listas deben tener el mismo largo y estar alineadas conforme al orden de las variables de tipo de derecho, naturaleza, su ejercicio y su uso (que serán entregadas antes del texto). 

**1. 'unidad_caudal'**
   - Campo único para todo el registro en base a la unidad explicitada en el documento, o que se puede inferir a partiri del lenguaje. 
   - "Litros por segundo" -> "L/S". "Metros cúbicos..." -> "M3/S". "Acciones" -> "ACCIONES", etc...

**2. 'tipo_caudal'**
   - "PERMANENTE": Si dice "permanente", "continuo" o similares
   - "EVENTUAL": Si dice "eventual", "sobrantes", "excedentes", "derrames" o similares.
   - "CAUDAL MÁXIMO INSTANTÁNEO": Sólo cuando se dice explícitamente dicho término.
   -  Si no es claro entonces utiliza "NO INDICA/OTRO"

**3. CAUDAL MENSUAL ('caudal_ene', 'caudal_feb'...) y 'caudal_promedio':**
   - Si tienes 2 derechos, y el Derecho 1 tiene tabla mensual pero el Derecho 2 es fijo (no varía), debes llenar la lista así: `[Valor_Mes_Derecho1, null]`.
   - El valor `null` (None) indica que ese derecho específico no tiene una valor particular para ese mes.
   - Si NINGÚN derecho tiene tabla mensual, deja los campos de meses vacíos (null) y completa sólamente el caudal promedio.
   - Si se declara un 'tipo_caudal' = "CAUDAL MÁXIMO INSTANTÁNEO", entonces en 'caudal_promedio' indica dicho valor. 

**3. FUENTE Y CUENCA ('tipo_fuente', 'fuente', 'cuenca'):**
   - **Nombre de la Fuente ('fuente'):** Extrae el nombre propio (ej: "Canal Las Mercedes", "Estero El Carmen", "Pozo N°4").
   - **Clasificación Técnica ('tipo_fuente'):** Debes clasificar la el nombre genérico de la fuente usando **SOLO** las opciones permitidas en tu lista de validación. Usa estas reglas de inferencia obligatoria:
   
   **REGLAS DE GATILLO:**
      1. Si la fuente contiene la palabra **"CANAL"**, **"ACEQUIA"** o **"ACUEDUCTO"** -> Clasifica SIEMPRE como **"CANAL"**.
      2. Si la fuente contiene **"RÍO"**, **"ESTERO"** -> Clasifica como **"RÍO / ESTERO"**.
      3. Si la fuente contiene **"POZO"**, **"NORIA"**, **"PUNTERA"** -> Clasifica como la variante subterránea específica.
      4. Si la fuente contiene **"VERTIENTE"** -> Clasifica como **"VERTIENTE"**.
      5. Si la fuente contiene **"EMBALSE"**, **"TRANQUE"**, **"LAGUNA"** -> Clasifica como **"EMBALSE / TRANQUE"** o **"LAGO / LAGUNA"**.
   - **Contexto de Naturaleza:** Utiliza el campo "NATURALEZA DEL AGUA" provisto en el encabezado para descartar opciones ilógicas (ej. si la naturaleza es SUBTERRÁNEA, no puedes elegir "RÍO").


NO ALUCINES. Si el texto no menciona explícitamente una fuente o caudal, pero tienes un número de derechos mayor, usa `null`.
"""         

### Modelo 5: Transacción

In [12]:
examples_5 = [
    {
        "input": """
            Don Juan Pérez vende a Inversiones Santa Marta SpA la Parcela 42 y sus derechos de agua...
            El precio de la compraventa es la suma alzada de cinco mil Unidades de Fomento, que se pagan al contado.
            Las partes declaran que del precio total, la suma de cuatro mil setecientas unidades de fomento corresponden al inmueble, 
            y el remanente se asignan al valor de los derechos de aprovechamiento de aguas aquí transferidos. 
            Los derechos registrales correspondientes a esta actuación ascienden a la suma de dos mil seicientos ($ 2.600) Pesos Chilenos. 
            """,
        "output": {
            "valor_transaccion": 5000.0,
            "valor_agua": 300.0,
            "unidad_transaccion": "UNIDAD DE FOMENTO (UF)"
        }
    },
    {
        "input": """
            [Cuerpo central:]
            ...don SERGIO EDUARDO ALBA FERNANDEZ, C.I. 12.265.291-2 y doña ROSA VERONICA ROBLES ARANDA, C.I. 7.240.520-K,
            son dueños de derechos de aprovechamiento de aguas, que consisten en cero coma cero veintisiete mil seiscientos veinticinco
            acciones del Canal Las Mercedes, y venden, ceden y transfieren la totalidad de dichos derechos de aprovechamiento 
            de aguas consuntivos a don SERGIO IDELFONSO ALBA CORRAL. El precio de la cesión es la cantidad de quince millones de pesos
            que el comprador paga a los vendedores en este acto, en dinero efectivo, a entera satisfacción de estos.
            No se señalan asignaciones de valor específicas por cuotas o derechos individuales.

            [Nota al margen derecho:]
            ...
            """,
        "output": {
            "valor_transaccion": 15000000.0,
            "valor_agua": 15000000.0, 
            "unidad_transaccion": "PESOS CHILENOS (CLP)"
        }
    }
]

instr_fase5 = """
### OBJETIVO ESPECÍFICO: Extraer el Valor de la Transacción

Tu objetivo es identificar el monto de la transacción y la moneda utilizada. 
Debes distinguir si existe un precio específico para el agua o si es un precio paquete (inmueble + agua).

### REGLAS DE EXTRACCIÓN:

**1. DETECCIÓN DE VALOR TOTAL DE LA TRANSACCIÓN ('valor_transaccion'):**
   - Busca frases como "en el precio de", "por la suma de", "valor de la compraventa".
   - Convierte palabras a números (ej. "diez millones doscientos mil " -> 1020000.0).
   - Si no hay monto en dinero, devuelve `null` en los valores.
   - NO buscamos el valor de los 'derechos registrales', ni ningún otro valor sobre el costo del trámite, aranceles, tarifas o tasas administrativas, sino que sobre el VALOR de la transacción.

**2. DETECCIÓN DE VALOR DEL AGUA ('valor_agua'):**
   - **Escenario A (Valor Separado):** Es posible que el texto desagregue el valor total de la transacción en el valor del derecho de agua y otros elementos.
       Si el texto dice: 
       - "asignando a los derechos de agua la suma de..."
       - "del precio total, corresponde a las aguas la suma de...", 
          o otros similares, extrae ese monto específico.
       - Debes corroborar que el valor_agua extraido sea siempre igual o inferior al valor_transaccion, sino estas cometiendo un error. 
   - **Escenario B (Valor Único):** Si el texto indica un precio total (ej. por la parcela y las aguas)
       y **NO** desagrega el valor específico del agua, debes asmir que el `valor_agua` es IGUAL al `valor_transaccion`.

**3. DETECCIÓN DE UNIDAD ('unidad_transaccion'):**
   - Identifica la moneda principal de la transacción, que puede estar expresada de distintas maneras, no necesariamente en dinero, por ejemplo:
     - "pesos", "$", "moneda nacional" -> `PESOS CHILENOS (CLP)`.
     - "US$, "USD", "dólares estadounidenses" -> `DÓLARES (USD)`.
     - "intercambio de canje de acciones" -> `ACCIONES`.
"""

### Modelo 6: Título Anterior

In [13]:
examples_6 = [
    {
        "input": """
            ...se encuentra inscrito a nombre de la vendedora a fojas 345 número 212 del Registro de Propiedad de Aguas del Conservador de Bienes Raíces de San Fernando del año 2015.
            El título anterior de dominio rola inscrito a fojas 10 vuelta número 8 del Registro de Propiedad de Aguas del mismo Conservador, año 1998...
            """,
        "output": {
            "ano_cbr_anterior": [2015, 1998],
            "numero_cbr_anterior": [212, 8],
            "fojas_cbr_anterior": ["345", "10v"],
            "cbr_anterior": ["SAN FERNANDO", "SAN FERNANDO"]
        }
    },
    {
        "input": """
            [Cuerpo central:]
            ...adquirió el inmueble por compraventa... inscrito a fojas 1.200 N° 800 del Registro de Propiedad del CBR de Buin año 2020.
            En cuanto a los derechos de aprovechamiento de aguas, estos rolan inscritos a fojas 55 vta. N° 40 del Registro de Propiedad de Aguas del Conservador de Bienes Raíces de Buin del año 2010.
            """,
        "output": {
            "ano_cbr_anterior": [2010],
            "numero_cbr_anterior": [40],
            "fojas_cbr_anterior": ["55v"],
            "cbr_anterior": ["BUIN"]
        }
    },
    {
        "input": """
            [Cuerpo central:]
            ...es dueño de los derechos de agua consistentes en 2 acciones del Canal La Punta...
            Los adquirió por herencia...
            El título de dominio se encuentra inscrito a fojas cuatrocientos, número doscientos del
            Registro de Propiedad de Aguas del año dos mil diecinueve a cargo del Conservador de Cauquenes.
            
            [Anotaciones manuscritas en el margen derecho]
            * Al lado de las líneas 1-2:
              > Hipoteca f. 5 N° 5 año 2002. [tachado] alzada [/tachado]
            * Al lado de las líneas 3-4:
              > hipoteca f. 4 N° 6 año 2002. [tachado] alzada [/tachado]
            * Al lado de las líneas 7-8:
              > Prohibición f. 2 vta N° 3 año 2004.-
            * En un recuadro, al lado de las líneas 11-16:
              > Transmisión Hereditaria
              > Ver Inscripción de
              > Fjs. 11 vta N° 11
              > del Registro del año 2016
              > Cauquenes, 16-03-2016
            """,
        "output": {
            "ano_cbr_anterior": [2019],
            "numero_cbr_anterior": [200],
            "fojas_cbr_anterior": ["400"],
            "cbr_anterior": ["CAUQUENES"]
        }
    }
]

instr_fase6 = """
### OBJETIVO: Extracción de Títulos Anteriores (Inscripciones Previas)
Debes identificar y extraer todas las inscripciones del **Registro de Propiedad de Aguas** que sean PREVIAS a la actual. 

### INFORMACIÓN DE REFERENCIA (NO EXTRAER):
Se te proporciona la **INSCRIPCIÓN ACTUAL** en el encabezado. 
- **REGLA DE ORO:** Si encuentras datos que coincidan exactamente con la "INSCRIPCIÓN ACTUAL" (Fojas, Número y Año), **IGNÓRALOS**. 
- Solo buscamos la cadena de títulos que dio origen a la actual.

### REGLAS CRÍTICAS DE FILTRADO:
1. **SOLO AGUAS:** 
   - **IGNORA** inscripciones de "Bienes Raíces", "Registro de Propiedad" (de suelo), "Hipotecas" o "Posesiones Efectivas", a menos que el texto diga explícitamente que están en el *Registro de Propiedad de Aguas*.
   - Si el texto no es explícito, busca palabras clave: "litros por segundo", "acciones de canal", "derechos de aprovechamiento", "La Merced".

2. **ORDEN CRONOLÓGICO ASCENDENTE (OBLIGATORIO):** - Las listas DEBEN ordenarse de la fecha más antigua a la más reciente (ej: `[1995, 2010, 2022]`).
   - Si los años coinciden, ordena por número de Fojas (menor a mayor).

3. **INTEGRIDAD:** No resumas. Si el documento menciona tres títulos anteriores distintos, extrae los tres.

4. **SINCRONIZACIÓN DE LISTAS:**
   - Si hay múltiples títulos de agua anteriores (ej. una lista de derechos), extrae todos los datos en orden cronológico.
   - Todas las listas (`ano`, `numero`, `fojas`, `cbr`) deben tener el mismo largo.


### REGLAS DE EXTRACCIÓN POR CAMPO:

**1. 'fojas_cbr_anterior' (List[str])**
   - Número de fojas. Si dice "vuelta" o "vta", normaliza a "v" (ej: "45 vuelta" -> "45v"). 

**2. 'numero_cbr_anterior' (List[str])**
   - Número de la inscripción en dígitos. Busca: "número Y", "N° Y", "nro Y".

**3. 'ano_cbr_anterior' (List[int])**
   - Año de la inscripción en dígitos. Busca: "año Z", "del Z". (ej: "dos mil tres" -> 2003).

**4. 'cbr_anterior' (List[str])**
   - Ciudad del Conservador ("del Archivero Judicial de la ciudad de XX")
   - **Importante:** Si el texto dice "del mismo Conservador" o "de este Conservador", utiliza el nombre del Conservador que aparece en el encabezado de este mensaje.
   - Si no se explicita, asume que es de la misma ciudad actual. 

### PISTAS DE CONTEXTO (ANCLAS):
- "El título anterior..."
- "Adquirió por inscripción de fojas..."
- "Inscripción que le antecede..."
- "Rola inscrito a...",
- "Antecedente: Registro de Propiedad de Aguas del año..."
"""

### Fase 7: Coordenadas

In [15]:
examples_7 = [
    {
        "input": """
            [Cuerpo central:]
            ...IDENTIFICACIÓN DE LAS AGUAS: derecho de aprovechamiento de aguas superficiales y detenidas...
            se extrae mecánicamente de las aguas del lago Ranco...
            en un punto de captación situado en la coordenada UTM (m): Norte: 5.552.929 y Este: 725.108, Datum WGS-84, huso 18.-
            """,
        "output": {
            "tipo_punto": ["PUNTO DE CAPTACIÓN"],
            "nombre_punto": ["Lago Ranco"],
            "norte": [5552929.0], 
            "este": [725108.0],  
            "datum": "WGS84",
            "huso": "18"
        }
    },
    {
        "input": """
            ...a fin de conducir las aguas, se constituye servidumbre de acueducto que gravará
            al predio sirviente en una franja de cinco metros de ancho.
            El trazado de la obra de conducción se inicia en la Bocatoma A ubicada en el Tranque Industrial estero Las Torcazas, 
            definida por las coordenadas geográficas treinta y ocho grados, dos minutos, treinta y nueve coma seis segundos 
            Latitud Sur y setenta y dos grados, cero minutos, cincuenta y siete coma seis segundos Longitud Oeste. 
            donde empalma con el canal matriz, cuyas coordenadas son treinta y ocho grados, dos minutos, cuarenta y cinco coma dos segundos 
            Latitud Sur y setenta y dos grados, un minuto, cinco coma tres segundos Longitud Oeste.
            """,
        "output": {
            "tipo_punto": ["PUNTO DE CAPTACIÓN", "PUNTO DE RESTITUCIÓN/DESCARGA"], 
            "nombre_punto": ["Bocatoma A Tranque Industrial", "Punto de Entrega"],
            "norte": [-38.0443, -38.0459], 
            "este": [-72.0160, -72.0181],  
            "datum": "WGS84", 
            "huso": "NO INDICA/OTRO"
        }
    }
]

instr_fase7 = """
### OBJETIVO ESPECÍFICO: Extracción y Normalización de Coordenadas

Extraer la ubicación geoespacial de los puntos de agua (captación, restitución, etc.) mencionados en el documento. 
Debes detectar si las coordenadas están en formato **UTM (Metros)** o en formato **GEOGRÁFICO (Grados/Minutos/Segundos)** 
y estandarizarlas usando los campos `norte` y `este`.

### REGLA MAESTRA DE ALMACENAMIENTO:

1. **SI SON UTM (Metros):** Son números grandes, positivos y sin decimales (generalmente).
   - Campo `norte`: Guarda el valor Norte (ej. 5400000).
   - Campo `este`: Guarda el valor Este (ej. 350000).

2. **SI SON GEOGRÁFICAS (Latitud/Longitud):**
   - Campo `norte`: Guarda la **LATITUD**.
   - Campo `este`: Guarda la **LONGITUD**.
   - **Conversión Obligatoria:** Si el texto está en Grados, Minutos y Segundos (escrito con números o palabras), debes convertirlo a **Grados Decimales**.
   - **Signo:** Si la latitud es "Sur" y la longitud es "Oeste" (lo normal en Chile), el valor final debe ser **NEGATIVO** (ej. -33.5).
   - *Fórmula de Conversión:* `Grados + (Minutos/60) + (Segundos/3600)` (multiplicar por -1 si es Sur/Oeste).

### INSTRUCCIONES DETALLADAS DE EXTRACCIÓN:

**1. Inferencia de 'tipo_punto' y Listas:**
   - Si el texto describe una servidumbre o trazado, generalmente:
     - el **primer punto** mencionado es el "PUNTO DE CAPTACIÓN" o "PUNTO DE RECEPCIÓN" 
     - el **último** es el "PUNTO DE RESTITUCIÓN/DESCARGA" o "PUNTO DE UTILIZACIÓN/CONSUMO".
   - Si el texto describe un cambio de punto de captación debes identificar como **"PUNTO DE CAPTACIÓN"** al punto original y cómo **"NUEVO PUNTO DE CAPTACIÓN"** el nuevo punto. 
   - Si aparecen coordenadas "sueltas" sin contexto claro, se puede inferir que es un **"PUNTO DE CAPTACIÓN"**.
   - Extrae tantos puntos como encuentres en orden de aparición.

**2. Manejo de Texto a Números:**
   - Es común que los números se escriban en palabras (ej: "treinta y ocho grados"). 
     Debes traducir estas palabras a dígitos antes de aplicar cualquier fórmula matemática o guardarlos.

**3. Inferencia de 'datum':**
   - Si encuentras coordenadas geográficas (Lat/Lon) y el texto **no especifica** un datum, asume y devuelve **"WGS84"** (estándar GPS).
   - Si son UTM y no se indica, usa "NO INDICA/OTRO".
   - Si se menciona explícitamente (PSAD56, SAD69, SIRGAS), usa ese valor.

**4. Limpieza:**
   - Ignora las coordenadas que definen los deslindes del predio (norte, sur, este, oeste del terreno). 
   - Solo extrae las coordenadas puntuales del **pozo, bocatoma, o trazado del agua**.
   - Asegurate de no estar repitiendo la misma coordenada geográfica dos veces. 

NO INVENTES INFORMACIÓN. SI NO HAY PUNTOS EXPLICITADOS, TODO QUEDA VACÍO. 
"""